In [1]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Localizar la raíz del proyecto
proyecto = Path.cwd()

if not (proyecto / "data" / "raw").is_dir():
    proyecto = proyecto.parent

ruta_internet = proyecto / "data" / "raw" / "internet.csv"

if not ruta_internet.is_file():
    raise FileNotFoundError(f"No se encontró: {ruta_internet}")

# Leer sin modificar el archivo original
internet = pd.read_csv(ruta_internet)

print("Dimensiones:", internet.shape)
print("\nColumnas:")
print(internet.columns.tolist())

print("\nInformación general:")
internet.info()

print("\nPrimeros registros:")
display(internet.head())


Dimensiones: (5517, 8)

Columnas:
['customerID', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

Información general:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5517 entries, 0 to 5516
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customerID        5517 non-null   object
 1   InternetService   5517 non-null   object
 2   OnlineSecurity    5517 non-null   object
 3   OnlineBackup      5517 non-null   object
 4   DeviceProtection  5517 non-null   object
 5   TechSupport       5517 non-null   object
 6   StreamingTV       5517 non-null   object
 7   StreamingMovies   5517 non-null   object
dtypes: object(8)
memory usage: 344.9+ KB

Primeros registros:


,customerID,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies
0,7590-VHVEG,DSL,No,Yes,No,No,No,No
1,5575-GNVDE,DSL,Yes,No,Yes,No,No,No
2,3668-QPYBK,DSL,Yes,Yes,No,No,No,No
3,7795-CFOCW,DSL,Yes,No,Yes,Yes,No,No
4,9237-HQITU,Fiber optic,No,No,No,No,No,No


In [2]:

print("Clientes únicos:", internet["customerID"].nunique())
print("IDs duplicados:", internet["customerID"].duplicated().sum())
print("Filas duplicadas:", internet.duplicated().sum())

print("\nValores nulos:")
display(internet.isna().sum())

print("\nCategorías por variable:")

for columna in internet.columns.drop("customerID"):
    print(f"\n{columna}")
    display(
        internet[columna]
        .value_counts(dropna=False)
        .rename_axis("Categoría")
        .to_frame("Clientes")
    )


Clientes únicos: 5517
IDs duplicados: 0
Filas duplicadas: 0

Valores nulos:


customerID          0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
dtype: int64


Categorías por variable:

InternetService


,Clientes
Categoría,
Fiber optic,3096
DSL,2421



OnlineSecurity


...

,Clientes
Categoría,
No,3473
Yes,2044



StreamingTV


,Clientes
Categoría,
No,2810
Yes,2707



StreamingMovies


,Clientes
Categoría,
No,2785
Yes,2732


In [3]:

# Comparar la contratación de servicios adicionales
# entre clientes DSL y Fiber optic.

servicios = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

# Convertimos Yes/No a indicadores únicamente para el cálculo.
# El DataFrame original internet no se modifica.
servicios_activos = internet[servicios].eq("Yes")

# Proporción de clientes que contratan cada servicio
# dentro de cada tecnología de internet.
adopcion_tecnologia = (
    servicios_activos
    .groupby(internet["InternetService"])
    .mean()
    .mul(100)
    .T
    .round(1)
)

adopcion_tecnologia["diferencia_pp"] = (
    adopcion_tecnologia["Fiber optic"]
    - adopcion_tecnologia["DSL"]
).round(1)

display(adopcion_tecnologia)


InternetService,DSL,Fiber optic,diferencia_pp
OnlineSecurity,48.7,27.1,-21.6
OnlineBackup,44.9,43.4,-1.5
DeviceProtection,44.0,43.8,-0.2
TechSupport,48.7,28.0,-20.7
StreamingTV,39.5,56.5,17.0
StreamingMovies,40.5,56.6,16.1


In [4]:

# Investigar pares de servicios contratados conjuntamente

pares_servicios = [
    ("OnlineSecurity", "TechSupport"),
    ("StreamingTV", "StreamingMovies")
]

resultados_pares = []

for tecnologia, grupo in internet.groupby("InternetService"):

    for servicio_a, servicio_b in pares_servicios:

        tiene_a = grupo[servicio_a].eq("Yes")
        tiene_b = grupo[servicio_b].eq("Yes")

        ambos = (tiene_a & tiene_b).sum()

        resultados_pares.append({
            "tecnologia": tecnologia,
            "servicio_a": servicio_a,
            "servicio_b": servicio_b,
            "clientes": len(grupo),
            "contratan_ambos": ambos,
            "porcentaje_ambos": round(
                100 * ambos / len(grupo), 1
            ),
            "esperado_independencia": round(
                100 * tiene_a.mean() * tiene_b.mean(), 1
            )
        })

coocurrencias = pd.DataFrame(resultados_pares)

display(coocurrencias)


,tecnologia,servicio_a,servicio_b,clientes,contratan_ambos,porcentaje_ambos,esperado_independencia
0,DSL,OnlineSecurity,TechSupport,2421,725,29.9,23.7
1,DSL,StreamingTV,StreamingMovies,2421,630,26.0,16.0
2,Fiber optic,OnlineSecurity,TechSupport,3096,374,12.1,7.6
3,Fiber optic,StreamingTV,StreamingMovies,3096,1310,42.3,32.0


In [5]:

# Número de servicios adicionales contratados por cliente

internet_perfiles = internet.copy()

internet_perfiles["numero_servicios"] = (
    internet_perfiles[servicios]
    .eq("Yes")
    .sum(axis=1)
)

# Distribución porcentual dentro de cada tecnología
distribucion_servicios = pd.crosstab(
    internet_perfiles["numero_servicios"],
    internet_perfiles["InternetService"],
    normalize="columns"
).mul(100).round(1)

display(distribucion_servicios)

# Resumen de cantidad de servicios
display(
    internet_perfiles
    .groupby("InternetService")["numero_servicios"]
    .describe()
    .round(2)
)


InternetService,DSL,Fiber optic
numero_servicios,,
0,12.1,12.9
1,17.1,17.8
2,18.8,18.7
3,19.6,20.8
4,14.8,15.9
5,11.6,9.4
6,5.9,4.6


,count,mean,std,min,25%,50%,75%,max
InternetService,,,,,,,,
DSL,2421.0,2.66,1.73,0.0,1.0,3.0,4.0,6.0
Fiber optic,3096.0,2.55,1.67,0.0,1.0,3.0,4.0,6.0


In [6]:

# Identificar las combinaciones exactas de servicios adicionales
# sin modificar el DataFrame original.

perfiles_internet = internet.copy()

perfiles_internet["perfil_servicios"] = (
    perfiles_internet[servicios]
    .eq("Yes")
    .apply(
        lambda fila: " + ".join(fila.index[fila])
        if fila.any() else "Sin servicios adicionales",
        axis=1
    )
)

# Frecuencia de cada combinación dentro de cada tecnología
combinaciones = (
    perfiles_internet
    .groupby(["InternetService", "perfil_servicios"])
    .size()
    .rename("clientes")
    .reset_index()
)

combinaciones["porcentaje"] = (
    100 * combinaciones["clientes"]
    / combinaciones.groupby("InternetService")["clientes"].transform("sum")
).round(2)

# Ocho combinaciones más frecuentes por tecnología
top_combinaciones = (
    combinaciones
    .sort_values(
        ["InternetService", "clientes"],
        ascending=[True, False]
    )
    .groupby("InternetService")
    .head(8)
)

display(top_combinaciones)

print(
    "Configuraciones distintas observadas:",
    perfiles_internet["perfil_servicios"].nunique()
)


,InternetService,perfil_servicios,clientes,porcentaje
56,DSL,Sin servicios adicionales,294,12.14
41,DSL,OnlineSecurity + OnlineBackup + DeviceProtecti...,143,5.91
24,DSL,OnlineSecurity,106,4.38
8,DSL,OnlineBackup,97,4.01
32,DSL,OnlineSecurity + DeviceProtection + TechSuppor...,79,3.26
45,DSL,OnlineSecurity + OnlineBackup + TechSupport,78,3.22
60,DSL,TechSupport,74,3.06
52,DSL,OnlineSecurity + TechSupport,68,2.81
120,Fiber optic,Sin servicios adicionales,399,12.89
123,Fiber optic,StreamingTV + StreamingMovies,181,5.85


Configuraciones distintas observadas: 64


### I5 — Configuraciones completas de los servicios de internet

**Población:** 5.517 clientes, distribuidos entre DSL (2.421) y Fiber optic (3.096).

Se investigaron las combinaciones de seis servicios adicionales contratados por cliente.

**Hallazgos:**

- La configuración individual más frecuente en ambas tecnologías es no contratar servicios adicionales: 12,14 % en DSL y 12,89 % en Fiber optic.
- Las ocho combinaciones más frecuentes representan el 38,79 % de los clientes DSL y el 45,68 % de los clientes Fiber optic.
- En DSL aparecen frecuentemente configuraciones que incluyen seguridad y soporte técnico.
- En Fiber optic destacan configuraciones asociadas con servicios de streaming.
- Ambas tecnologías presentan una mediana de tres servicios adicionales, pese a las diferencias en su composición.

**Conclusión:** los clientes presentan configuraciones diversas. La composición de servicios parece más informativa para distinguir tecnologías que la cantidad de complementos contratados.

**Limitaciones:** las asociaciones son descriptivas. No conocemos el uso efectivo de los servicios ni su relación con el abandono a partir de esta tabla aislada.

**Decisión metodológica:** conservar las seis variables originales y la característica derivada `numero_servicios` como candidatas para el posterior análisis integrado. No realizar clustering adicional en esta etapa.

**Estado:** exploración estructural de servicios completada.